In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
orders = (
spark.read\
.format("csv")\
.options(header=True, inferSchema=True)\
.load("/Volumes/business_to_business/sports_bar_data/orders")
)

In [0]:
orders.count()

9183

In [0]:
orders.write\
.format("delta")\
.option("delta.enableChangeDataFeed", True)\
.mode("append")\
.saveAsTable("business_to_business.bronze.orders")

## Stanging the newly added records


In [0]:
orders\
.write\
.format("delta")\
.option("delta.enableChangeDataFeed", True)\
.mode("overwrite")\
.saveAsTable("business_to_business.bronze.stag_orders")

In [0]:
files = dbutils.fs.ls("/Volumes/business_to_business/sports_bar_data/orders")
print(files)
for file in files:
    dbutils.fs.mv(
        file.path,
        "/Volumes/business_to_business/sports_bar_data/processed_orders/" + file.name
    )

[FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_03.csv', name='orders_2025_12_03.csv', size=21899, modificationTime=1788841709000), FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_04.csv', name='orders_2025_12_04.csv', size=22385, modificationTime=1788841710000), FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_05.csv', name='orders_2025_12_05.csv', size=19859, modificationTime=1788841710000), FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_06.csv', name='orders_2025_12_06.csv', size=20158, modificationTime=1788841710000), FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_07.csv', name='orders_2025_12_07.csv', size=20837, modificationTime=1788841710000), FileInfo(path='dbfs:/Volumes/business_to_business/sports_bar_data/orders/orders_2025_12_08.csv', name='orders_2025_12_08.csv', size=21711, 

### Silver Layer

In [0]:
df_orders = spark.read.table("business_to_business.bronze.stag_orders").withColumn("ingestion_timestamp", F.current_timestamp())\
.select("*", "_metadata.file_name", "_metadata.file_size")
display(df_orders)

order_id,order_placement_date,customer_id,product_id,order_qty,ingestion_timestamp,file_name,file_size
FDEC85101603,"Wednesday, December 03, 2025",789101,25891301,92.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",789101,25891502,232.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,03/12/2025,789101,25891402,478.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",789101,25891201,349.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",789101,25891602,62.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",789101,88888888,484.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",ABC987,25891603,64.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85101603,"Wednesday, December 03, 2025",789101,25891402,478.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85301402,"Wednesday, December 03, 2025",789301,25891203,428.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968
FDEC85301402,03-12-2025,789301,25891302,48.0,2026-09-08T04:29:14.899Z,part-00000-f5b97ef0-da5b-4212-8d66-3ea62d2ae2a4.c000.zstd.parquet,53968


In [0]:
from pyspark.sql import functions as F

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-12-03|2025-12-30|
+----------+----------+



In [0]:
df_products = spark.table("business_to_business.silver.dim_sbproducts")
orders_df = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

orders_df.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty| ingestion_timestamp|           file_name|file_size|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
|FDEC85301402|          2025-12-03|     789301|  25891103|    348.0|2026-09-08 04:29:...|part-00000-f5b97e...|    53968|6c7000a2708d3a2ec...|
|FDEC84221601|          2025-12-03|     789221|  25891601|    144.0|2026-09-08 04:29:...|part-00000-f5b97e...|    53968|38b61b697918c0ad7...|
|FDEC84221601|          2025-12-03|     789221|  25891101|    368.0|2026-09-08 04:29:...|part-00000-f5b97e...|    53968|521fcd441ab9d975c...|
|FDEC85220601|          2025-12-03|     789220|  25891403|    408.0|2026-09-08 04:29:...|part-00000-f5b97e...|    53968|53361f1d15f3967db...|
|FDEC8

In [0]:
from delta.tables import DeltaTable

In [0]:
if not (spark.catalog.tableExists("business_to_business.silver.orders")):
    orders_df.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable("business_to_business.silver.orders")
else:
    silver_delta = DeltaTable.forName(spark, "business_to_business.silver.orders")
    silver_delta.alias("silver").merge(orders_df.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
orders_df.write\
.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", True)\
.saveAsTable("business_to_business.silver.stg_orders")

### Gold Layer

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM business_to_business.silver.stg_orders;")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FDEC85301402|2025-12-03|       789301|6c7000a2708d3a2ec...|  25891103|        348.0|
|FDEC84221601|2025-12-03|       789221|38b61b697918c0ad7...|  25891601|        144.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists("business_to_business.gold.orders")):
    df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema", True)\
    .saveAsTable("business_to_business.gold.fact_sborders")
else:
    gold_delta = DeltaTable.forName(spark, "business_to_business.gold.fact_sborders")
    gold_delta.alias("gold").merge(
        source = df_gold.alias("s"),
        condition= "gold.order_id = s.order_id AND gold.date = s.date AND gold.customer_code = s.customer_code AND gold.product_code = s.product_code"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


### Merge with the Parent Company

In [0]:

df_child =  spark.sql(f"SELECT order_placement_date as date FROM business_to_business.silver.stg_orders")

incremental_month_df = df_child.select(
    F.trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

+-----------+
|start_month|
+-----------+
| 2025-12-01|
+-----------+



In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM business_to_business.gold.fact_sborders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

Total Rows:  7215
+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-12-03|6c7000a2708d3a2ec...|       789301|        348.0|
|2025-12-03|38b61b697918c0ad7...|       789221|        144.0|
|2025-12-03|521fcd441ab9d975c...|       789221|        368.0|
|2025-12-03|53361f1d15f3967db...|       789220|        408.0|
|2025-12-03|52d9158987029d4fa...|       789320|         41.0|
|2025-12-03|a09b63cb03fb789b4...|       789303|        296.0|
|2025-12-03|521fcd441ab9d975c...|       789621|        250.0|
|2025-12-03|a09b63cb03fb789b4...|       789503|        469.0|
|2025-12-03|d84e9b44b1ae0668a...|       789601|        384.0|
|2025-12-03|a09b63cb03fb789b4...|       789702|        221.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
monthly_table.show(5)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-12-03|6c7000a2708d3a2ec...|       789301|        348.0|
|2025-12-03|38b61b697918c0ad7...|       789221|        144.0|
|2025-12-03|521fcd441ab9d975c...|       789221|        368.0|
|2025-12-03|53361f1d15f3967db...|       789220|        408.0|
|2025-12-03|52d9158987029d4fa...|       789320|         41.0|
+----------+--------------------+-------------+-------------+
only showing top 5 rows


In [0]:
df_monthly_table = (monthly_table.
                    withColumn("month_start", F.trunc("date", "MM"))
                    .groupBy("customer_code", "product_code", "month_start")
                    .agg(F.sum("sold_quantity").alias("sold_quantity"))
                    .withColumnRenamed("month_start", "date")
                    
)


In [0]:
df_monthly_table.count()

612

In [0]:
gold_parent_data = DeltaTable.forName(spark, "business_to_business.gold.fact_orders")
gold_parent_data.alias("t").merge(
    source=df_monthly_table.alias("s"),
    condition="t.customer_code = s.customer_code AND t.product_code = s.product_code AND t.date = s.date"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
drop table business_to_business.bronze.stag_orders

In [0]:
%sql
drop table business_to_business.silver.stg_orders 